In [ ]:
!pip install networkx pandas matplotlib -q

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

stations = pd.read_csv('/content/sample_data/islamabad_metrobus_stations.csv')
interchanges = pd.read_csv('/content/sample_data/islamabad_metrobus_interchanges.csv')

print(stations.shape, interchanges.shape)
stations.head()

In [ ]:
G = nx.Graph()

# Give each station a unique node ID per line (line + sequence),
# but store the real station name as an attribute
for _, row in stations.iterrows():
    node_id = f"{row['line']}_{row['sequence']}"
    G.add_node(node_id, name=row['station_name'], line=row['line'])

# Add edges: consecutive stations on the same line
for line, group in stations.groupby('line'):
    group = group.sort_values('sequence')
    node_ids = [f"{line}_{seq}" for seq in group['sequence']]
    for i in range(len(node_ids) - 1):
        G.add_edge(node_ids[i], node_ids[i+1], type='same_line')

print("Nodes before merging interchanges:", G.number_of_nodes())
print("Edges before merging interchanges:", G.number_of_edges())

In [ ]:
# Refined Fix for Data Alignment with unique column names
import pandas as pd
import networkx as nx

# 1. Reload and ensure unique column mapping
interchanges_final = pd.read_csv('/content/sample_data/islamabad_metrobus_interchanges.csv')
interchanges_final = interchanges_final.reset_index()

# Explicitly name columns to avoid duplicates like 'line_b'
# Based on the observed shift: index -> station, station_name -> line_a, line_a -> line_b
assigned_cols = ['station_name', 'line_a', 'line_b']
remaining_count = len(interchanges_final.columns) - len(assigned_cols)
new_cols = assigned_cols + [f'extra_{i}' for i in range(remaining_count)]
interchanges_final.columns = new_cols

def find_node_by_line_and_name(line, name):
    if pd.isna(line) or pd.isna(name):
        return None
    line_s = str(line).strip().lower()
    name_s = str(name).strip().lower()
    match = stations[(stations['line'].str.strip().str.lower() == line_s) &
                     (stations['station_name'].str.strip().str.lower() == name_s)]
    if match.empty:
        return None
    return f"{match.iloc[0]['line']}_{match.iloc[0]['sequence']}"

# 2. Re-initialize Graph and merge
G = nx.Graph()
for _, row in stations.iterrows():
    node_id = f"{row['line']}_{row['sequence']}"
    G.add_node(node_id, name=row['station_name'], line=row['line'])

for line, group in stations.groupby('line'):
    group = group.sort_values('sequence')
    node_ids = [f"{line}_{seq}" for seq in group['sequence']]
    for i in range(len(node_ids) - 1):
        G.add_edge(node_ids[i], node_ids[i+1], type='same_line')

node_representative = {node_id: node_id for node_id in G.nodes()}

print("Merging components with unique column mapping...")
for _, row in interchanges_final.iterrows():
    s_name = row['station_name']
    l_a = row['line_a']
    l_b = row['line_b']

    # Ensure we are checking scalar values
    if pd.isna(s_name) or pd.isna(l_a) or pd.isna(l_b):
        continue

    n_a = find_node_by_line_and_name(l_a, s_name)
    n_b = find_node_by_line_and_name(l_b, s_name)

    if n_a and n_b:
        curr_a = node_representative[n_a]
        curr_b = node_representative[n_b]

        if curr_a != curr_b:
            G = nx.contracted_nodes(G, curr_a, curr_b, self_loops=False)
            for node, rep in node_representative.items():
                if rep == curr_b:
                    node_representative[node] = curr_a
            print(f"Success: Connected {s_name} ({l_a} <-> {l_b})")

print(f"\nFinal node count: {G.number_of_nodes()}")
print(f"Connected components: {nx.number_connected_components(G)}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 1. Define a color map based on your line names
def get_node_color(line_name):
    line_name = str(line_name).lower()
    if 'red' in line_name: return '#e63946'
    if 'blue' in line_name: return '#1d3557'
    if 'green' in line_name: return '#2a9d8f'
    if 'orange' in line_name: return '#f4a261'
    return '#8d99ae' # Default gray for unknown lines

# 2. Extract colors and labels
node_colors = []
labels = {}
for node, data in G.nodes(data=True):
    # When nodes merge, they keep the 'line' attribute of the first node (line_a)
    color = get_node_color(data.get('line', ''))
    node_colors.append(color)
    labels[node] = data.get('name', str(node))

# 3. Setup the plot
plt.figure(figsize=(16, 12))
plt.title("Islamabad Metrobus Network", fontsize=22, fontweight='bold', pad=20)

# 4. Generate layout (Kamada-Kawai places path-graphs very naturally)
# If this layout throws an error due to disconnected components, use nx.spring_layout(G, k=0.15)
try:
    pos = nx.kamada_kawai_layout(G)
except:
    pos = nx.spring_layout(G, k=0.15, iterations=50)

# 5. Draw edges and nodes
nx.draw_networkx_edges(G, pos, edge_color='gray', width=2, alpha=0.7)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=150, edgecolors='white', linewidths=1.5)

# 6. Draw labels (tweak font_size if it gets too cluttered)
nx.draw_networkx_labels(G, pos, labels, font_size=8, font_weight='bold',
                        verticalalignment='bottom')

# Optional: Add a custom legend
red_patch = mpatches.Patch(color='#e63946', label='Red Line')
blue_patch = mpatches.Patch(color='#1d3557', label='Blue Line')
green_patch = mpatches.Patch(color='#2a9d8f', label='Green Line')
orange_patch = mpatches.Patch(color='#f4a261', label='Orange Line')
plt.legend(handles=[red_patch, blue_patch, green_patch, orange_patch], loc='best', fontsize=12)

plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
degree_c = nx.degree_centrality(G)
betweenness_c = nx.betweenness_centrality(G)
closeness_c = nx.closeness_centrality(G)

# node names got merged into a 'contraction' attribute list by networkx;
# grab a clean display name for each remaining node
def get_display_name(node):
    return G.nodes[node].get('name', str(node))

centrality_df = pd.DataFrame({
    'node_id': list(G.nodes()),
    'station_name': [get_display_name(n) for n in G.nodes()],
    'degree': [degree_c[n] for n in G.nodes()],
    'betweenness': [betweenness_c[n] for n in G.nodes()],
    'closeness': [closeness_c[n] for n in G.nodes()],
})

centrality_df.sort_values('betweenness', ascending=False).head(15)

In [ ]:
def global_efficiency(graph):
    return nx.global_efficiency(graph)

baseline_efficiency = global_efficiency(G)
print("Baseline global efficiency:", baseline_efficiency)

criticality_scores = []
for node in G.nodes():
    H = G.copy()
    H.remove_node(node)
    if H.number_of_nodes() == 0:
        continue

    eff_after = global_efficiency(H)
    largest_cc = len(max(nx.connected_components(H), key=len)) if H.number_of_nodes() > 0 else 0
    frag_ratio = largest_cc / H.number_of_nodes() if H.number_of_nodes() > 0 else 0
    efficiency_drop = (baseline_efficiency - eff_after) / baseline_efficiency if baseline_efficiency > 0 else 0

    criticality_scores.append({
        'node_id': node,
        'station_name': get_display_name(node),
        'efficiency_drop_pct': efficiency_drop * 100,
        'largest_component_fraction': frag_ratio
    })

criticality_df = pd.DataFrame(criticality_scores).sort_values('efficiency_drop_pct', ascending=False)
criticality_df.head(20)

In [ ]:
final_df = centrality_df.merge(criticality_df, on=['node_id', 'station_name'])
final_df = final_df.sort_values('efficiency_drop_pct', ascending=False)
final_df.to_csv('islamabad_metrobus_criticality_results.csv', index=False)
final_df.head(20)

In [ ]:
top_n = 15
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_between = final_df.sort_values('betweenness', ascending=False).head(top_n)
axes[0].barh(top_between['station_name'], top_between['betweenness'], color='steelblue')
axes[0].set_title('Top Stations by Betweenness Centrality')
axes[0].invert_yaxis()

top_sub = final_df.sort_values('efficiency_drop_pct', ascending=False).head(top_n)
axes[1].barh(top_sub['station_name'], top_sub['efficiency_drop_pct'], color='indianred')
axes[1].set_title('Top Stations by Substitute-Network Criticality (% Efficiency Drop)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('criticality_comparison.png', dpi=200)
plt.show()

In [ ]:
plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G, seed=42, k=0.4)

node_sizes = [4000 * betweenness_c[n] + 30 for n in G.nodes()]
nx.draw_networkx_edges(G, pos, alpha=0.4, width=1)
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='crimson', alpha=0.8)

# Label only the top betweenness stations to avoid clutter
top_labels = set(top_between['node_id'])
labels = {n: get_display_name(n) for n in G.nodes() if n in top_labels}
nx.draw_networkx_labels(G, pos, labels, font_size=8)

plt.title("Islamabad-Rawalpindi Metrobus Network")
plt.axis('off')
plt.savefig('islamabad_metrobus_network.png', dpi=200)
plt.show()